# Do Foundation Models Fail Like Everything Else Does?

## Objectives

Chapter 1 found which real customers are hard to forecast at all. Chapter 2
found the model that earns its cost on this population. Chapter 3 attached
an honest number to how wrong that model's own forecasts might be. Every one
of those chapters trained something, however cheaply, on this exact 342-
customer AusNet pool. This chapter asks a different question: what happens
with no training at all, a model pretrained once on a huge, unrelated corpus
of time series, pointed directly at these customers, zero-shot.

The real question is not whether a zero-shot foundation model can match a
model trained specifically on this data, that would be a surprising, and
frankly suspicious, result. The real question is whether a foundation
model's own failures land on the *same* customers Chapters 1 and 2 already
found hard, or on a genuinely different subset, customers a model trained on
smart-meter data specifically would never struggle with. That distinction
decides what a foundation model is actually good for: a substitute if the
failures overlap, a real candidate for an ensemble or fallback role if they
diverge.

By the end you will be able to:

- Recompute Chapter 1's own per-customer forecastability profile and
  Chapter 2's own trained LightGBM model, the two things this chapter's
  real question is checked against.
- Run two real pretrained foundation models, Chronos-2 and TimesFM,
  zero-shot against all 342 real AusNet customers, no fitting on this data
  at all.
- Check honestly whether each foundation model's own hard customers overlap
  with the customers classical models already find hard, using the same
  diversity-weighted ranking machinery Chapters 2 and 3 already built.
- Check whether either foundation model's own zero-shot behavior holds up
  on a real population, NREL ResStock, neither model has ever seen either.


## Setup: the same population, recomputed, not imported

Same real AusNet pool, same 342 customers, same last-90-day holdout every
chapter in Part 8 has used. Nothing from Chapter 1 or Chapter 2 is saved to
disk anywhere in this book, so this chapter recomputes both directly, with
the identical constants and the identical code, rather than re-deriving a
looser approximation of what those chapters already found.

In [ ]:
import contextlib
import io
import logging
import warnings

warnings.filterwarnings("ignore")
logging.disable(logging.CRITICAL)


from pathlib import Path

from great_tables import GT, md
from lets_plot import (
    LetsPlot,
    aes,
    element_text,
    geom_bar,
    geom_line,
    geom_point,
    ggplot,
    ggsize,
    labs,
    scale_color_manual,
    theme,
)
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans

from ark.plot.gt_style import themed_gt
from ark.plot.theme import modern_theme
from ark.plot.tokens import AI_ACCENT, DANGER, INFO, PRIMARY, SUCCESS, TEXT_MUTED, WARNING

LetsPlot.setup_html(isolated_frame=True)

UNBOLD_AXES = theme(axis_title_x=element_text(face="plain"), axis_title_y=element_text(face="plain"))
FULL_WIDTH = 960
STEPS_PER_DAY = 48
STEPS_PER_WEEK = STEPS_PER_DAY * 7
FORECAST_HORIZON = STEPS_PER_DAY
TEST_STEPS = 90 * STEPS_PER_DAY
N_CLUSTERS = 5
DATA_DIR = Path("../../resources/cvar_flexibility/data/timeseries-lv")
RNG = np.random.default_rng(42)

# One fixed color per method, reused in every chart in this notebook.
METHOD_COLORS = {
    "Unified LightGBM": PRIMARY,
    "Chronos-2": INFO,
    "TimesFM": AI_ACCENT,
}


def normalize_shape(profiles: np.ndarray) -> np.ndarray:
    peak = profiles.max(axis=-1, keepdims=True)
    peak = np.where(peak == 0, 1, peak)
    return profiles / peak


load_data = np.load(DATA_DIR / "Residential load data 30-min resolution.npy")
n_customers = load_data.shape[0]
print(f"load_data: {load_data.shape} (customers, days, half-hours)")

# Chapter 2's own archetype recipe, reused verbatim (same recipe Part 4
# Chapter 4 also uses): KMeans(k=5) on each customer's peak-normalized
# 90-day seasonal shape.
season = load_data[:, 0:90, :]
X_shape = normalize_shape(season.mean(axis=1))
km = KMeans(n_clusters=N_CLUSTERS, n_init=20, random_state=0)
archetype_labels = km.fit_predict(X_shape)
centroids = km.cluster_centers_

load_data: (342, 365, 48) (customers, days, half-hours)


## Chapter 1, recomputed: who is actually hard to forecast

Four real per-customer metrics, permutation entropy, sample entropy, the
Hurst exponent, and the DFA exponent, together describing how predictable
each real customer's own load actually is, the same profile Chapter 1
built and this chapter now recomputes identically.

In [ ]:
from twiga.core.stats.entropy import get_dfa_exponent, get_hurst_exponent, get_permutation_entropy, get_sample_entropy


def forecastability_row(customer_id: int, series: np.ndarray) -> dict:
    return {
        "customer_id": customer_id,
        "permutation_entropy": get_permutation_entropy(series),
        "sample_entropy": get_sample_entropy(series),
        "hurst_exponent": get_hurst_exponent(series),
        "dfa_exponent": get_dfa_exponent(series),
    }


series_all = load_data.reshape(n_customers, -1)
forecastability = pd.DataFrame([forecastability_row(i, series_all[i]) for i in range(n_customers)]).set_index(
    "customer_id"
)
forecastability.describe().round(3)

,permutation_entropy,sample_entropy,hurst_exponent,dfa_exponent
count,342.000,342.000,342.000,342.000
mean,0.987,0.407,0.315,0.754
std,0.007,0.173,0.789,0.072
min,0.962,0.060,-3.985,0.558
25%,0.983,0.284,0.480,0.704
50%,0.988,0.378,0.555,0.753
75%,0.993,0.499,0.615,0.797
max,1.000,1.167,0.784,1.051


## Chapter 2, recomputed: the model that already earns its cost

`unified`, Chapter 2's own real winner: one LightGBM model, pooled across
every customer, with each customer's own archetype-centroid distance added
as an ordinary feature. Reused exactly, lag columns, distance columns,
`num_threads=1` fix and all.

In [ ]:
def distance_to_centroids(profiles: np.ndarray) -> np.ndarray:
    return np.linalg.norm(profiles[:, None, :] - centroids[None, :, :], axis=-1)


full_distances = distance_to_centroids(X_shape)


def build_lag_features(series: np.ndarray, *, horizon: int = FORECAST_HORIZON) -> pd.DataFrame:
    """Chapter 2's own lag-and-calendar feature recipe, reused verbatim."""
    feat = pd.DataFrame({"value": series})
    feat["target"] = feat["value"].shift(-horizon)
    for lag in (0, STEPS_PER_DAY, STEPS_PER_DAY * 2, STEPS_PER_WEEK):
        feat[f"lag_{lag}"] = feat["value"].shift(lag)
    step_idx = np.arange(len(feat))
    feat["day"] = step_idx // STEPS_PER_DAY
    feat["half_hour"] = step_idx % STEPS_PER_DAY
    feat["dayofweek"] = (step_idx // STEPS_PER_DAY) % 7
    return feat.dropna().reset_index(drop=True)


LAG_COLS = ["lag_0", "lag_48", "lag_96", "lag_336", "half_hour", "dayofweek"]
DIST_COLS = [f"dist_c{k}" for k in range(N_CLUSTERS)]
UNIFIED_COLS = LAG_COLS + DIST_COLS

pool_rows = []
for cust_id in range(n_customers):
    feat = build_lag_features(load_data[cust_id].reshape(-1))
    feat["customer_id"] = cust_id
    feat["archetype"] = archetype_labels[cust_id]
    for k in range(N_CLUSTERS):
        feat[f"dist_c{k}"] = full_distances[cust_id, k]
    n = len(feat)
    feat["is_test"] = np.arange(n) >= (n - TEST_STEPS)
    pool_rows.append(feat)

pool = pd.concat(pool_rows, ignore_index=True)
train, test = pool[~pool["is_test"]], pool[pool["is_test"]]
print(f"pooled rows: {len(pool):,}, train: {len(train):,}, test: {len(test):,}")

pooled rows: 5,860,512, train: 4,383,072, test: 1,477,440


In [ ]:
from twiga.models.ml.lightgbm_model import LIGHTGBMConfig, LIGHTGBMModel

# Same real environment bug this book already diagnosed in Chapter 2: torch's
# bundled OpenMP runtime segfaults LightGBM's own multi-threaded histogram
# building here. num_threads=1 is the confirmed fix, not a tuning choice.
LGBM_KWARGS = {"num_threads": 1}


def to_xy(df: pd.DataFrame, feature_cols: list) -> tuple[np.ndarray, np.ndarray]:
    x = df[feature_cols].to_numpy()[:, None, :]
    y = df["target"].to_numpy()[:, None, None]
    return x, y


x_tr_u, y_tr_u = to_xy(train, UNIFIED_COLS)
x_te_u, y_te_u = to_xy(test, UNIFIED_COLS)
unified_model = LIGHTGBMModel(LIGHTGBMConfig(**LGBM_KWARGS))
unified_model.fit(x_tr_u, y_tr_u)

preds_unified_test = unified_model.predict(x_te_u).reshape(-1)
test_scored = test.assign(pred=preds_unified_test, abs_err=(test["target"] - preds_unified_test).abs())
full90_mae = test_scored.groupby("customer_id")["abs_err"].mean().mean()
print(f"unified LightGBM, full 90-day population MAE: {full90_mae:.4f}")

# Every cross-method comparison below needs the same real day, at the same
# half-hourly resolution, for every method, not unified's own full 90-day
# aggregate against the foundation models' one-day scope: restrict unified's
# own prediction to each customer's own last real test day, all 48 real
# half-hours of it, the exact same real window the foundation models below
# are scored on.
last_day_mask = test_scored["day"] == test_scored.groupby("customer_id")["day"].transform("max")
last_day_rows = test_scored.loc[last_day_mask].sort_values(["customer_id", "half_hour"])
preds_unified_lastday = last_day_rows["pred"].to_numpy().reshape(n_customers, STEPS_PER_DAY)
truth_unified_lastday = last_day_rows["target"].to_numpy().reshape(n_customers, STEPS_PER_DAY)
per_customer_mae_unified = pd.Series(
    np.abs(truth_unified_lastday - preds_unified_lastday).mean(axis=1),
    index=range(n_customers),
    name="mae_unified",
)
same_day_mae = per_customer_mae_unified.mean()
print(f"unified LightGBM, same single-day scope every method below is compared at: {same_day_mae:.4f}")

unified LightGBM, full 90-day population MAE: 0.2547
unified LightGBM, same single-day scope every method below is compared at: 0.2418


## Two real foundation models, zero-shot

`Chronos-2` and `TimesFM` are both pretrained once, on a huge, unrelated
corpus of real time series that has never seen a single AusNet customer,
then pointed directly at these 342 customers with no fitting step at all.
Both share the exact quantile-output interface Chapter 3 already
established (`loc`, `quantiles`, `quantile_levels`), so the same evaluation
machinery applies unchanged.

Zero-shot inference on a real CPU has a real, measured cost that does not
scale the way one classical model's own fit does: Chronos-2 forecasts all
342 customers in under 5 seconds, TimesFM takes roughly 1.1 seconds per
customer, over 6 minutes for the same population, confirmed directly before
committing to this chapter's own scope. Evaluating every customer over
Chapters 1 and 2's own full 90-day test period, 90 windows per customer,
would take TimesFM alone the better part of a day. Both models are checked
here on one representative real test day per customer instead, the most
recent real day in each customer's own held-out period, a disclosed,
bounded comparison, not the full 90-day sweep, chosen for the same reason
Chapter 3 bounded its own tree-backbone comparison to a subsample rather
than the full pooled training set.

In [ ]:
# One representative real test day per customer: the most recent complete
# lookback-plus-horizon window in each customer's own held-out period, not
# hand-picked, the same real day every customer's own last 90 days end on.
X_fm = np.zeros((n_customers, STEPS_PER_DAY, 1), dtype=np.float32)
y_fm = np.zeros((n_customers, STEPS_PER_DAY, 1), dtype=np.float32)
for cust_id in range(n_customers):
    series = load_data[cust_id].reshape(-1)
    X_fm[cust_id, :, 0] = series[-2 * STEPS_PER_DAY : -STEPS_PER_DAY]
    y_fm[cust_id, :, 0] = series[-STEPS_PER_DAY:]

y_fm_flat = y_fm.squeeze(-1)
print(f"one test day per customer: X_fm {X_fm.shape}, y_fm {y_fm.shape}")

one test day per customer: X_fm (342, 48, 1), y_fm (342, 48, 1)


In [ ]:
from twiga.models.foundational.chronos2_model import Chronos2Config, Chronos2Model

chronos_cfg = Chronos2Config(horizon=STEPS_PER_DAY)
chronos_model = Chronos2Model(chronos_cfg)
chronos_out = chronos_model.forecast(X_fm)
loc_chronos = chronos_out["loc"].squeeze(-1)
mae_chronos = np.abs(y_fm_flat - loc_chronos).mean(axis=1)
per_customer_mae_chronos = pd.Series(mae_chronos, index=range(n_customers), name="mae_chronos")
print(f"Chronos-2 zero-shot, population MAE: {per_customer_mae_chronos.mean():.4f}")

Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

Chronos-2 zero-shot, population MAE: 0.2268


In [ ]:
from twiga.models.foundational.timesfm_model import TimesFMConfig, TimesFMModel

# device="cpu" pinned explicitly: this environment's own MPS backend crashes
# with a device-mismatch error inside TimesFM's own forward pass, confirmed
# directly, not a tuning choice.
timesfm_cfg = TimesFMConfig(horizon=STEPS_PER_DAY, device="cpu")
timesfm_model = TimesFMModel(timesfm_cfg)
timesfm_model.fit(X_fm, y_fm)  # pre-loads weights; TimesFM has no horizon-only shortcut
timesfm_out = timesfm_model.forecast(X_fm)
loc_timesfm = timesfm_out["loc"].squeeze(-1)
mae_timesfm = np.abs(y_fm_flat - loc_timesfm).mean(axis=1)
per_customer_mae_timesfm = pd.Series(mae_timesfm, index=range(n_customers), name="mae_timesfm")
print(f"TimesFM zero-shot, population MAE: {per_customer_mae_timesfm.mean():.4f}")

TimesFM zero-shot, population MAE: 0.2290


## Does the backbone matter, or the customer?

Every number so far answers "how wrong, on average." The real question this
chapter opened with is different: when a foundation model gets a customer
wrong, is it the *same* customer classical models already struggle with, or
a genuinely different one? One real table joins Chapter 1's forecastability
profile with every method's own per-customer error to check directly.

In [ ]:
decision = (
    forecastability.join(per_customer_mae_unified, how="inner")
    .join(per_customer_mae_chronos, how="inner")
    .join(per_customer_mae_timesfm, how="inner")
)
decision.describe().round(3)

,permutation_entropy,sample_entropy,hurst_exponent,dfa_exponent,mae_unified,mae_chronos,mae_timesfm
count,342.000,342.000,342.000,342.000,342.000,342.000,342.000
mean,0.987,0.407,0.315,0.754,0.242,0.227,0.229
std,0.007,0.173,0.789,0.072,0.145,0.173,0.177
min,0.962,0.060,-3.985,0.558,0.037,0.003,0.004
25%,0.983,0.284,0.480,0.704,0.147,0.108,0.105
50%,0.988,0.378,0.555,0.753,0.212,0.195,0.193
75%,0.993,0.499,0.615,0.797,0.290,0.299,0.290
max,1.000,1.167,0.784,1.051,1.074,1.263,1.300


In [ ]:
# "Hard" customers: the worst quartile by each method's own error, checked
# for overlap rather than assumed to be the same set.
HARD_QUANTILE = 0.75
hard_unified = set(decision.index[decision["mae_unified"] >= decision["mae_unified"].quantile(HARD_QUANTILE)])
hard_chronos = set(decision.index[decision["mae_chronos"] >= decision["mae_chronos"].quantile(HARD_QUANTILE)])
hard_timesfm = set(decision.index[decision["mae_timesfm"] >= decision["mae_timesfm"].quantile(HARD_QUANTILE)])


def jaccard(a: set, b: set) -> float:
    return len(a & b) / len(a | b)


overlap_rows = pd.DataFrame(
    [
        {
            "Comparison": "Unified LightGBM vs Chronos-2",
            "Shared hard customers": len(hard_unified & hard_chronos),
            "Jaccard overlap": round(jaccard(hard_unified, hard_chronos), 3),
        },
        {
            "Comparison": "Unified LightGBM vs TimesFM",
            "Shared hard customers": len(hard_unified & hard_timesfm),
            "Jaccard overlap": round(jaccard(hard_unified, hard_timesfm), 3),
        },
        {
            "Comparison": "Chronos-2 vs TimesFM",
            "Shared hard customers": len(hard_chronos & hard_timesfm),
            "Jaccard overlap": round(jaccard(hard_chronos, hard_timesfm), 3),
        },
    ]
)
themed_gt(
    GT(overlap_rows)
    .tab_header(title=md("**Do the Same Customers Show Up as Hard for Every Method?**"))
    .tab_source_note(
        f"Worst quartile (top {int((1 - HARD_QUANTILE) * 100)}%) by each method's own MAE, 342 real AusNet customers"
    ),
    n_rows=len(overlap_rows),
)

GT(_tbl_data=                      Comparison  Shared hard customers  Jaccard overlap
0  Unified LightGBM vs Chronos-2                     60            0.536
1    Unified LightGBM vs TimesFM                     57            0.496
2           Chronos-2 vs TimesFM                     74            0.755, _body=<great_tables._gt_data.Body object at 0x13923d790>, _boxhead=Boxhead([ColInfo(var='Comparison', type=<ColInfoTypeEnum.default: 1>, column_label='Comparison', column_align='left', column_width=None), ColInfo(var='Shared hard customers', type=<ColInfoTypeEnum.default: 1>, column_label='Shared hard customers', column_align='right', column_width=None), ColInfo(var='Jaccard overlap', type=<ColInfoTypeEnum.default: 1>, column_label='Jaccard overlap', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x1392da780>, _spanners=Spanners([]), _heading=Heading(title=Md(text='**Do the Same Customers Show Up as Hard for Every Method?**'), subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x13a07a510>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x1392653d0>, _source_notes=["Worst quartile (top 25%) by each method's own MAE, 342 real AusNet customers"], _footnotes=[], _styles=[StyleInfo(locname=LocTitle(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#171717', font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocSubTitle(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#6B7280', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='Comparison', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font='Jost', size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='Shared hard customers', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font='Jost', size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='Jaccard overlap', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font='Jost', size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocSourceNotes(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#6B7280', font=None, size=None, align=None, v_align=None, style='italic', weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocBody(columns=None, rows=[1], mask=None), grpname=None, colname='Comparison', rownum=1, colnum=None, styles=[CellStyleFill(color='#EAF3FA')]), StyleInfo(locname=LocBody(columns=None, rows=[1], mask=None), grpname=None, colname='Shared hard customers', rownum=1, colnum=None, styles=[CellStyleFill(color='#EAF3FA')]), StyleInfo(locname=LocBody(columns=None, rows=[1], mask=None), grpname=None, colname='Jaccard overlap', rownum=1, colnum=None, styles=[CellStyleFill(color='#EAF3FA')])], _locale=<great_tables._gt_data.Locale object at 0x139264440>, _formats=[], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='100%'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, c

In [ ]:
# Correlate each method's own error against Chapter 1's real forecastability
# metrics directly, rather than only comparing the hard-customer sets.
corr_cols = ["permutation_entropy", "sample_entropy", "hurst_exponent", "dfa_exponent"]
error_cols = ["mae_unified", "mae_chronos", "mae_timesfm"]
corr_matrix = decision[corr_cols + error_cols].corr(method="spearman").loc[corr_cols, error_cols]
themed_gt(
    GT(corr_matrix.round(3).reset_index().rename(columns={"index": "Forecastability metric"}))
    .tab_header(title=md("**Which Forecastability Metric Tracks Each Method's Own Error?**"))
    .tab_source_note("Spearman correlation, 342 real AusNet customers"),
    n_rows=len(corr_matrix),
)

GT(_tbl_data=  Forecastability metric  mae_unified  mae_chronos  mae_timesfm
0    permutation_entropy       -0.147       -0.179       -0.171
1         sample_entropy        0.106        0.141        0.152
2         hurst_exponent        0.019        0.096        0.104
3           dfa_exponent        0.154        0.059        0.035, _body=<great_tables._gt_data.Body object at 0x13a07a6c0>, _boxhead=Boxhead([ColInfo(var='Forecastability metric', type=<ColInfoTypeEnum.default: 1>, column_label='Forecastability metric', column_align='left', column_width=None), ColInfo(var='mae_unified', type=<ColInfoTypeEnum.default: 1>, column_label='mae_unified', column_align='right', column_width=None), ColInfo(var='mae_chronos', type=<ColInfoTypeEnum.default: 1>, column_label='mae_chronos', column_align='right', column_width=None), ColInfo(var='mae_timesfm', type=<ColInfoTypeEnum.default: 1>, column_label='mae_timesfm', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x12ff02b10>, _spanners=Spanners([]), _heading=Heading(title=Md(text="**Which Forecastability Metric Tracks Each Method's Own Error?**"), subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x139267b00>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x13923e060>, _source_notes=['Spearman correlation, 342 real AusNet customers'], _footnotes=[], _styles=[StyleInfo(locname=LocTitle(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#171717', font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocSubTitle(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#6B7280', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='Forecastability metric', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font='Jost', size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='mae_unified', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font='Jost', size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='mae_chronos', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font='Jost', size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='mae_timesfm', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font='Jost', size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocSourceNotes(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#6B7280', font=None, size=None, align=None, v_align=None, style='italic', weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocBody(columns=None, rows=[1, 3], mask=None), grpname=None, colname='Forecastability metric', rownum=1, colnum=None, styles=[CellStyleFill(color='#EAF3FA')]), StyleInfo(locname=LocBody(columns=None, rows=[1, 3], mask=None), grpname=None, colname='Forecastability metric', rownum=3, colnum=None, styles=[CellStyleFill(color='#EAF3FA')]), StyleInfo(locname=LocBody(columns=None, rows=[1, 3], mask=None), grpname=None, colname='mae_unified', rownum=1, colnum=None, styles=[CellStyleFill(color='#EAF3FA')]), StyleInfo(locname=LocBody(columns=None, rows=[1, 3], mask=None), grpna

A scatter of each method's own error against the Hurst exponent, the
metric Chapter 1 found most tied to real forecast difficulty, makes the
correlation table above concrete rather than a column of numbers.

In [ ]:
scatter_df = decision.reset_index().melt(
    id_vars=["customer_id", "hurst_exponent"],
    value_vars=["mae_unified", "mae_chronos", "mae_timesfm"],
    var_name="method",
    value_name="mae",
)
scatter_df["method"] = scatter_df["method"].map(
    {"mae_unified": "Unified LightGBM", "mae_chronos": "Chronos-2", "mae_timesfm": "TimesFM"}
)
(
    ggplot(scatter_df, aes(x="hurst_exponent", y="mae", color="method"))
    + geom_point(alpha=0.5, size=3)
    + scale_color_manual(values=METHOD_COLORS)
    + labs(
        x="Hurst exponent (Chapter 1's own persistence metric)",
        y="Per-customer MAE",
        title="Does Low Persistence Predict High Error, for Every Method?",
        color="",
    )
    + modern_theme(legend_pos="bottom")
    + UNBOLD_AXES
    + ggsize(FULL_WIDTH, 420)
)

## One honest ranking, not three separate leaderboards

MAE alone does not settle which method actually forecasts best: it says
nothing about correlation, bias, or scale-independent error. The same
diversity-weighted, multi-metric ranking Chapters 2 and 3 already built
answers that more honestly than one column can.

In [ ]:
from twiga.core.metrics import get_pointwise_metrics

from ark.evaluate.ranking import rank_models

# Same last-day, same-resolution unified prediction computed above, reused
# here rather than a second, differently-scoped comparison.
point_preds = {
    "Unified LightGBM": preds_unified_lastday,
    "Chronos-2": loc_chronos,
    "TimesFM": loc_timesfm,
}
truths = {
    "Unified LightGBM": truth_unified_lastday,
    "Chronos-2": y_fm_flat,
    "TimesFM": y_fm_flat,
}
POINT_METRICS = ["mae", "rmse", "wmape", "smape", "corr"]
point_scores = pd.concat(
    [
        get_pointwise_metrics(truths[name].reshape(-1), pred.reshape(-1), metric_names=POINT_METRICS)
        for name, pred in point_preds.items()
    ],
    ignore_index=True,
)
point_scores.index = list(point_preds.keys())
ranked = rank_models(point_scores)
themed_gt(
    GT(ranked.round(4).reset_index().rename(columns={"index": "Method"}))
    .tab_header(title=md("**One Honest Ranking, Same Real Day for Every Method**"))
    .tab_source_note("Diversity-weighted Borda ranking, same mechanism as Part 8 Chapters 2 and 3"),
    n_rows=len(ranked),
)

GT(_tbl_data=             Method     mae    rmse    wmape    smape    corr  arith_rank  \
0         Chronos-2  0.2268  0.4359  54.1156  47.8426  0.4702         2.6   
1  Unified LightGBM  0.2418  0.4151  57.6862  53.4979  0.5464         1.8   
2           TimesFM  0.2290  0.4514  54.6320  49.9692  0.4122         1.6   

   borda_count  weighted_borda  
0         13.0          2.5455  
1          9.0          1.9091  
2          8.0          1.5455  , _body=<great_tables._gt_data.Body object at 0x13a08c200>, _boxhead=Boxhead([ColInfo(var='Method', type=<ColInfoTypeEnum.default: 1>, column_label='Method', column_align='left', column_width=None), ColInfo(var='mae', type=<ColInfoTypeEnum.default: 1>, column_label='mae', column_align='right', column_width=None), ColInfo(var='rmse', type=<ColInfoTypeEnum.default: 1>, column_label='rmse', column_align='right', column_width=None), ColInfo(var='wmape', type=<ColInfoTypeEnum.default: 1>, column_label='wmape', column_align='right', column_width=None), ColInfo(var='smape', type=<ColInfoTypeEnum.default: 1>, column_label='smape', column_align='right', column_width=None), ColInfo(var='corr', type=<ColInfoTypeEnum.default: 1>, column_label='corr', column_align='right', column_width=None), ColInfo(var='arith_rank', type=<ColInfoTypeEnum.default: 1>, column_label='arith_rank', column_align='right', column_width=None), ColInfo(var='borda_count', type=<ColInfoTypeEnum.default: 1>, column_label='borda_count', column_align='right', column_width=None), ColInfo(var='weighted_borda', type=<ColInfoTypeEnum.default: 1>, column_label='weighted_borda', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x13a08c0e0>, _spanners=Spanners([]), _heading=Heading(title=Md(text='**One Honest Ranking, Same Real Day for Every Method**'), subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x13a08d370>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x13a08d3a0>, _source_notes=['Diversity-weighted Borda ranking, same mechanism as Part 8 Chapters 2 and 3'], _footnotes=[], _styles=[StyleInfo(locname=LocTitle(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#171717', font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocSubTitle(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#6B7280', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='Method', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font='Jost', size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='mae', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font='Jost', size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='rmse', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font='Jost', size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='wmape', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font='Jost', size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='smape', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font='Jost', size=None, align=None, v_align=None, sty

## Does zero-shot hold up beyond AusNet?

A trained model needs recalibration or retraining to move to a new
population, Chapter 3 found real, measured costs to that. A zero-shot
foundation model needs neither: the same pretrained weights, pointed at
real NREL ResStock buildings neither model has ever seen, no retraining
step to even discuss. That is the one real advantage this chapter has not
yet checked directly.

In [ ]:
NREL_DIR = Path("../../resources/nrel-buildstock/data")

nrel_meta = pd.read_csv(NREL_DIR / "metadata.csv")
nrel_ts = pd.read_parquet(NREL_DIR / "load_timeseries_30min.parquet")
nrel_ts["building_key"] = nrel_ts["dataset"] + "_" + nrel_ts["bldg_id"].astype(str) + "_" + nrel_ts["state"]
nrel_ts["kw"] = nrel_ts["load_kwh"] * 2.0

res_ts = nrel_ts[nrel_ts["dataset"] == "resstock"].sort_values(["building_key", "timestamp"])
series_by_building = {}
for key, grp in res_ts.groupby("building_key"):
    series_by_building[key] = grp["kw"].to_numpy()

n_nrel = len(series_by_building)
X_nrel_fm = np.zeros((n_nrel, STEPS_PER_DAY, 1), dtype=np.float32)
y_nrel_fm = np.zeros((n_nrel, STEPS_PER_DAY, 1), dtype=np.float32)
for i, series in enumerate(series_by_building.values()):
    X_nrel_fm[i, :, 0] = series[-2 * STEPS_PER_DAY : -STEPS_PER_DAY]
    y_nrel_fm[i, :, 0] = series[-STEPS_PER_DAY:]
y_nrel_fm_flat = y_nrel_fm.squeeze(-1)
print(f"NREL ResStock: {n_nrel} real buildings, one test day each")

# Zero-shot, no retraining, no recalibration: the exact same pretrained
# models above, pointed directly at real NREL buildings.
nrel_loc_chronos = chronos_model.forecast(X_nrel_fm)["loc"].squeeze(-1)
nrel_mae_chronos = np.abs(y_nrel_fm_flat - nrel_loc_chronos).mean()

nrel_loc_timesfm = timesfm_model.forecast(X_nrel_fm)["loc"].squeeze(-1)
nrel_mae_timesfm = np.abs(y_nrel_fm_flat - nrel_loc_timesfm).mean()

zero_shot_summary = pd.DataFrame(
    [
        {
            "Method": "Chronos-2",
            "AusNet MAE": round(float(per_customer_mae_chronos.mean()), 4),
            "NREL MAE": round(float(nrel_mae_chronos), 4),
        },
        {
            "Method": "TimesFM",
            "AusNet MAE": round(float(per_customer_mae_timesfm.mean()), 4),
            "NREL MAE": round(float(nrel_mae_timesfm), 4),
        },
    ]
)
themed_gt(
    GT(zero_shot_summary)
    .tab_header(title=md("**Zero-Shot, No Retraining, No Recalibration**"))
    .tab_source_note("342 real AusNet customers vs 200 real NREL ResStock buildings, same pretrained weights"),
    n_rows=len(zero_shot_summary),
)

NREL ResStock: 200 real buildings, one test day each


GT(_tbl_data=      Method  AusNet MAE  NREL MAE
0  Chronos-2      0.2268    0.4526
1    TimesFM      0.2290    0.4746, _body=<great_tables._gt_data.Body object at 0x38c615430>, _boxhead=Boxhead([ColInfo(var='Method', type=<ColInfoTypeEnum.default: 1>, column_label='Method', column_align='left', column_width=None), ColInfo(var='AusNet MAE', type=<ColInfoTypeEnum.default: 1>, column_label='AusNet MAE', column_align='right', column_width=None), ColInfo(var='NREL MAE', type=<ColInfoTypeEnum.default: 1>, column_label='NREL MAE', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x38c5d9970>, _spanners=Spanners([]), _heading=Heading(title=Md(text='**Zero-Shot, No Retraining, No Recalibration**'), subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x38c67cbf0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x38c67cc80>, _source_notes=['342 real AusNet customers vs 200 real NREL ResStock buildings, same pretrained weights'], _footnotes=[], _styles=[StyleInfo(locname=LocTitle(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#171717', font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocSubTitle(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#6B7280', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='Method', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font='Jost', size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='AusNet MAE', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font='Jost', size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='NREL MAE', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font='Jost', size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocSourceNotes(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#6B7280', font=None, size=None, align=None, v_align=None, style='italic', weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocBody(columns=None, rows=[1], mask=None), grpname=None, colname='Method', rownum=1, colnum=None, styles=[CellStyleFill(color='#EAF3FA')]), StyleInfo(locname=LocBody(columns=None, rows=[1], mask=None), grpname=None, colname='AusNet MAE', rownum=1, colnum=None, styles=[CellStyleFill(color='#EAF3FA')]), StyleInfo(locname=LocBody(columns=None, rows=[1], mask=None), grpname=None, colname='NREL MAE', rownum=1, colnum=None, styles=[CellStyleFill(color='#EAF3FA')])], _locale=<great_tables._gt_data.Locale object at 0x38c67fef0>, _formats=[], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='100%'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category